# Merge QLoRA adapter into Qwen2.5-VL-3B and export GGUF for Ollama

Chain: fp16 base -> apply LoRA -> merge -> convert to GGUF -> quantise Q4_K_M.

Only the quantised model and the vision projector are kept in the output, so the
local pull stays around 2-3GB instead of the ~15GB of intermediates.

In [ ]:
import os, subprocess, sys, shutil, json
from pathlib import Path

BASE     = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER  = None          # resolved below from the attached dataset
MERGED   = Path("/kaggle/temp/merged")
WORK     = Path("/kaggle/working")
LLAMA    = Path("/kaggle/temp/llama.cpp")

# Intermediates go in /kaggle/temp, NOT /kaggle/working: anything left in
# working becomes kernel output and would make the download ~15GB.
MERGED.parent.mkdir(parents=True, exist_ok=True)

for root in Path("/kaggle/input").glob("*"):
    hit = list(root.rglob("adapter_config.json"))
    if hit:
        ADAPTER = hit[0].parent
        break
assert ADAPTER is not None, "no adapter_config.json under /kaggle/input"
print("adapter:", ADAPTER)
print(json.dumps({k: v for k, v in json.load(open(ADAPTER / "adapter_config.json")).items()
                  if k in ("base_model_name_or_path", "r", "lora_alpha", "task_type")}, indent=2))

In [ ]:
# Pinned so a surprise upstream release cannot change behaviour mid-hackathon.
!pip -q install "transformers==4.51.3" "peft==0.14.0" "accelerate==1.3.0" 2>&1 | tail -2
print("installed")

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

# CPU + fp16 keeps peak RAM near the 6.2GB weight size. Kaggle has ~30GB, so
# this is comfortable here even though it is exactly what fails on the laptop.
print("loading base (this pulls ~7.5GB the first time)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True)
processor = AutoProcessor.from_pretrained(BASE)
print("base loaded:", sum(p.numel() for p in model.parameters()) / 1e9, "B params")

In [ ]:
model = PeftModel.from_pretrained(model, str(ADAPTER), torch_dtype=torch.float16)

# VERIFY the adapter actually attached. The adapter was trained against
# unsloth's 4-bit copy of this base; if module naming ever diverged, PEFT would
# match nothing, merge would be a no-op, and we would ship the stock model
# believing it was fine-tuned. Fail loudly instead.
lora_mods = [n for n, _ in model.named_modules() if "lora_A" in n]
print(f"LoRA modules injected: {len(lora_mods)}")
assert lora_mods, "adapter matched ZERO modules - refusing to export the base as 'fine-tuned'"
print("sample:", lora_mods[:3])

In [ ]:
print("merging...")
model = model.merge_and_unload()
MERGED.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MERGED, safe_serialization=True)
processor.save_pretrained(MERGED)
del model
import gc; gc.collect()
print("merged ->", MERGED)
!du -sh {MERGED}

In [ ]:
# llama.cpp supplies both the HF->GGUF converter and the quantiser.
if not LLAMA.exists():
    !git clone -q --depth 1 https://github.com/ggerganov/llama.cpp {LLAMA}
!pip -q install -r {LLAMA}/requirements/requirements-convert_hf_to_gguf.txt 2>&1 | tail -1
print("llama.cpp ready")

In [ ]:
F16 = Path("/kaggle/temp/qwen25vl-3b-ahc-f16.gguf")
# Two artefacts come out of a vision model: the language GGUF and the mmproj
# vision projector. --mmproj writes the projector on its own pass.
!python {LLAMA}/convert_hf_to_gguf.py {MERGED} --outfile {F16} --outtype f16
!python {LLAMA}/convert_hf_to_gguf.py {MERGED} --outfile {WORK}/mmproj-qwen25vl-3b-f16.gguf --mmproj
!ls -la {F16} {WORK}/mmproj-qwen25vl-3b-f16.gguf

In [ ]:
# Q4_K_M matches the quantisation of the stock ollama qwen2.5vl:3b this replaces,
# so latency and VRAM stay where they were measured (27-45s/call on a GTX 1650).
Q4 = WORK / "qwen25vl-3b-ahc-q4_k_m.gguf"
!cmake -S {LLAMA} -B {LLAMA}/build -DLLAMA_CURL=OFF > /dev/null 2>&1 && \
 cmake --build {LLAMA}/build --target llama-quantize -j4 > /dev/null 2>&1
QUANT = f"{LLAMA}/build/bin/llama-quantize"
!{QUANT} {F16} {Q4} Q4_K_M
!ls -la {WORK}

In [ ]:
# The Modelfile ships with the weights so the local step is a single command
# and nobody has to remember the projector line.
(WORK / "Modelfile").write_text(
    "FROM ./qwen25vl-3b-ahc-q4_k_m.gguf\n"
    "FROM ./mmproj-qwen25vl-3b-f16.gguf\n"
    'PARAMETER temperature 0.1\n'
    'PARAMETER num_ctx 4096\n'
)

# Guard the download size: anything stray left in /kaggle/working is output.
print("FINAL OUTPUT:")
tot = 0
for f in sorted(WORK.rglob("*")):
    if f.is_file():
        tot += f.stat().st_size
        print(f"  {f.name:<44} {f.stat().st_size/1e9:6.2f} GB")
print(f"  {'TOTAL':<44} {tot/1e9:6.2f} GB")